# Behovskartan 2 - DuckDB-Based Energy Demand Generator

This notebook generates energy demand forecasts for Sweden using Energy Agency projections as the base.

## DuckDB Pipeline Architecture

Uses pure SQL operations for 10-100x performance improvement over pandas:

1. **Load** annual demand from DuckDB (3 scenarios, 7 segments, 21 counties)
2. **Aggregate** transport sub-segments into single `transport` segment (7→5 segments)
3. **Create** profile lookup tables in DuckDB (pattern-based and year-based)
4. **Generate** timestamp backbone using `generate_series()` (2025-2050, 228k hours)
5. **Extend** profiles via SQL JOINs with weekday-aware matching
6. **Add noise and normalize** profiles per (segment, year) - noise BEFORE normalization guarantees exact annual totals
7. **Join** normalized profiles with annual demand
8. **Output** partitioned parquet files using `COPY TO`

## Data Sources

- **Annual Demand**: `input/energy_agency_scenarios/energy_agency.duckdb`
- **Profiles**: `input/load_profiles/*.csv` (year-based), `*.json` (pattern-based)

## Setup & Configuration

In [1]:
import duckdb
import json
from pathlib import Path
import time

# === WORKFLOW CONTROL ===
REBUILD_BASE = True  # Set True to regenerate base timeseries from scratch

# Paths
GENERATOR_PATH = Path.cwd().parent
INPUT_PATH = GENERATOR_PATH / 'input'
OUTPUT_PATH = GENERATOR_PATH / 'output'
PROFILE_PATH = INPUT_PATH / 'load_profiles'
DB_PATH = INPUT_PATH / 'energy_agency_scenarios' / 'energy_agency.duckdb'

# Configuration
CONFIG = {
    'start_year': 2025,
    'end_year': 2050,
    'noise_amplitude': 0.02,  # ±2% per year
    'noise_seed': 0.42,       # DuckDB setseed value
    'compression': 'zstd',
}

# === TIME TRACKING ===
# Centralized timing tracker for performance monitoring
TIMING = {
    'base_start': None,
    'base_end': None,
    'scenarios_start': None,
    'scenarios_end': None,
    'scenario_times': [],  # List of (scenario_name, elapsed_seconds)
}

print(f"Generator path: {GENERATOR_PATH}")
print(f"Input path: {INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"DuckDB source: {DB_PATH}")
print(f"Source exists: {DB_PATH.exists()}")
print(f"\nREBUILD_BASE: {REBUILD_BASE}")

Generator path: /home/viktor/code/behovskartan/generator
Input path: /home/viktor/code/behovskartan/generator/input
Output path: /home/viktor/code/behovskartan/generator/output
DuckDB source: /home/viktor/code/behovskartan/generator/input/energy_agency_scenarios/energy_agency.duckdb
Source exists: True

REBUILD_BASE: True


## Profile Configuration

Define which profiles map to which segments, and their file types.

In [2]:
# Segment to profile file mapping
# Transport segments use pattern profiles; housing/services/industry use year profiles
SEGMENT_PROFILES = {
    # Year-based: Full 8760-hour profiles from CSVs
    'housing': {'type': 'year', 'file': 'profile_housing_south_2024.csv'},
    'services': {'type': 'year', 'file': 'profile_services_south_2024.csv'},
    'industry': {'type': 'year', 'file': 'profile_industry_south_2024.csv'},
    
    # Pattern-based: hourly × weekday × monthly JSON patterns
    'datacenters': {'type': 'pattern', 'file': 'profile_datacenters_patterns.json'},
    'transport_cars': {'type': 'pattern', 'file': 'profile_transport_cars_patterns.json'},
    'transport_trucks': {'type': 'pattern', 'file': 'profile_transport_trucks_patterns.json'},
    'transport_rail': {'type': 'pattern', 'file': 'profile_transport_rail_patterns.json'},
}

# Final aggregated segments (transport sub-segments will be combined)
AGGREGATED_SEGMENTS = ['housing', 'services', 'industry', 'transport', 'datacenters']
TRANSPORT_SEGMENTS = ['transport_cars', 'transport_trucks', 'transport_rail']

print("Profile configuration:")
for seg, cfg in SEGMENT_PROFILES.items():
    file_exists = (PROFILE_PATH / cfg['file']).exists()
    status = "✓" if file_exists else "✗"
    print(f"  {status} {seg}: {cfg['file']} ({cfg['type']})")

Profile configuration:
  ✓ housing: profile_housing_south_2024.csv (year)
  ✓ services: profile_services_south_2024.csv (year)
  ✓ industry: profile_industry_south_2024.csv (year)
  ✓ datacenters: profile_datacenters_patterns.json (pattern)
  ✓ transport_cars: profile_transport_cars_patterns.json (pattern)
  ✓ transport_trucks: profile_transport_trucks_patterns.json (pattern)
  ✓ transport_rail: profile_transport_rail_patterns.json (pattern)


## Skip Logic

Determine whether to regenerate base timeseries or load from existing parquet output.

In [3]:
# Expected scenarios from Energy Agency
EXPECTED_SCENARIOS = ['Beslutad Policy', 'Internationell Tillväxt', 'Lokal Miljöhänsyn']

def base_needs_rebuild():
    """Check if base generation should run."""
    if REBUILD_BASE:
        print("REBUILD_BASE=True, regenerating base timeseries...")
        return True
    
    # Check for new nested structure: base/{scenario}/data.parquet
    base_dir = OUTPUT_PATH / 'base'
    if not base_dir.exists():
        print("No base/ directory found, generating...")
        return True
    
    # Check each expected scenario has a parquet file
    missing_scenarios = []
    for scenario in EXPECTED_SCENARIOS:
        scenario_file = base_dir / scenario / 'data.parquet'
        if not scenario_file.exists():
            missing_scenarios.append(scenario)
    
    if missing_scenarios:
        print(f"Missing scenarios: {missing_scenarios}, regenerating...")
        return True
    
    print(f"✓ Base timeseries exists ({len(EXPECTED_SCENARIOS)} scenarios). Skipping generation.")
    print(f"  Set REBUILD_BASE=True to force regeneration.")
    return False

RUN_BASE_GENERATION = base_needs_rebuild()

REBUILD_BASE=True, regenerating base timeseries...


In [4]:
# Load from parquet if base generation is skipped
# This cell provides the DuckDB connection for scenario work

if not RUN_BASE_GENERATION:
    print("Loading from parquet...")
    start = time.time()
    
    # Connect to in-memory DuckDB
    con = duckdb.connect(':memory:')
    
    # Create view to nested parquet files (base/{scenario}/{segment}/data.parquet)
    # Rename scenario_id -> scenario to match the rest of the notebook
    con.execute(f"""
        CREATE VIEW hourly_demand AS 
        SELECT 
            timestamp,
            value,
            geography,
            segment,
            scenario_id as scenario
        FROM read_parquet('{OUTPUT_PATH}/base/*/*/data.parquet')
    """)
    
    # Load annual_demand from source database
    con.execute(f"ATTACH '{DB_PATH}' AS source (READ_ONLY)")
    con.execute("""
        CREATE VIEW annual_demand AS 
        SELECT
            scenario,
            CASE WHEN segment IN ('transport_cars', 'transport_trucks', 'transport_rail') THEN 'transport' ELSE segment END as segment,
            geography, year, SUM(value) as value
        FROM source.annual_demand
        GROUP BY scenario, CASE WHEN segment IN ('transport_cars', 'transport_trucks', 'transport_rail') THEN 'transport' ELSE segment END, geography, year
    """)
    
    # Show stats
    count = con.execute("SELECT COUNT(*) FROM hourly_demand").fetchone()[0]
    scenarios = con.execute("SELECT DISTINCT scenario FROM hourly_demand ORDER BY scenario").fetchall()
    segments = con.execute("SELECT DISTINCT segment FROM hourly_demand ORDER BY segment").fetchall()
    print(f"✓ Loaded from parquet in {time.time()-start:.2f}s")
    print(f"  hourly_demand: {count:,} rows")
    print(f"  Scenarios: {[s[0] for s in scenarios]}")
    print(f"  Segments: {[s[0] for s in segments]}")
    print(f"  Parquet path: {OUTPUT_PATH}/base/*/*/data.parquet")
    print(f"\n>>> Skip to 'Summary' section or add scenario operations below <<<")
else:
    print(">>> Run the following cells to generate base timeseries <<<")

>>> Run the following cells to generate base timeseries <<<


---

## Step 0: Load Annual Demand and Aggregate Transport

Load data from energy_agency.duckdb and aggregate transport sub-segments (cars, trucks, rail) into a single `transport` segment for efficiency.

In [5]:
# Skip if loading from cache
if not RUN_BASE_GENERATION:
    print("Skipping base generation (loaded from cache)")
    print(">>> Jump to Summary section <<<")
else:
    # Start timing for base generation
    TIMING['base_start'] = time.time()
    
    # Connect to in-memory DuckDB (we'll load data from source and keep it in memory)
    con = duckdb.connect(':memory:')

    # Load annual demand from source database
    print(f"Loading annual demand from: {DB_PATH}")
    start = time.time()

    con.execute(f"""
        ATTACH '{DB_PATH}' AS source (READ_ONLY);
        
        -- Load original 7-segment data
        CREATE TABLE annual_demand_raw AS
        SELECT * FROM source.annual_demand;
    """)

    # Show original data stats
    result = con.execute("""
        SELECT 
            COUNT(*) as rows,
            COUNT(DISTINCT scenario) as scenarios,
            COUNT(DISTINCT segment) as segments,
            COUNT(DISTINCT geography) as geographies,
            MIN(year) as min_year,
            MAX(year) as max_year
        FROM annual_demand_raw
    """).fetchone()

    print(f"Loaded {result[0]:,} rows in {time.time()-start:.2f}s")
    print(f"  Scenarios: {result[1]}")
    print(f"  Segments: {result[2]}")
    print(f"  Geographies: {result[3]}")
    print(f"  Years: {result[4]}-{result[5]}")

Loading annual demand from: /home/viktor/code/behovskartan/generator/input/energy_agency_scenarios/energy_agency.duckdb
Loaded 11,466 rows in 0.02s
  Scenarios: 3
  Segments: 7
  Geographies: 21
  Years: 2025-2050


In [6]:
# Show segments before aggregation
print("Segments before aggregation:")
print(con.execute("""
    SELECT segment, SUM(value) as total_gwh
    FROM annual_demand_raw
    GROUP BY segment
    ORDER BY total_gwh DESC
""").fetchdf().to_string(index=False))

Segments before aggregation:
         segment    total_gwh
        industry 8.285851e+06
         housing 3.022723e+06
        services 2.443331e+06
  transport_cars 7.654755e+05
     datacenters 5.418987e+05
transport_trucks 5.024508e+05
  transport_rail 2.821095e+05


In [7]:
# Create transport weights based on relative demand across all data
# This will be used to create a weighted average transport profile
con.execute("""
    CREATE TABLE transport_weights AS
    WITH transport_totals AS (
        SELECT 
            segment,
            SUM(value) as segment_total
        FROM annual_demand_raw
        WHERE segment IN ('transport_cars', 'transport_trucks', 'transport_rail')
        GROUP BY segment
    ),
    grand_total AS (
        SELECT SUM(segment_total) as total FROM transport_totals
    )
    SELECT 
        t.segment,
        t.segment_total / g.total as weight
    FROM transport_totals t, grand_total g
""")

print("Transport segment weights (for weighted average profile):")
print(con.execute("SELECT * FROM transport_weights").fetchdf().to_string(index=False))

Transport segment weights (for weighted average profile):
         segment   weight
  transport_cars 0.493844
  transport_rail 0.182002
transport_trucks 0.324154


In [8]:
# Aggregate transport segments into single 'transport' segment
# This reduces segments from 7 to 5, improving performance by ~30%
con.execute("""
    CREATE TABLE annual_demand AS
    SELECT
        scenario,
        CASE
            WHEN segment IN ('transport_cars', 'transport_trucks', 'transport_rail')
            THEN 'transport'
            ELSE segment
        END as segment,
        geography,
        year,
        SUM(value) as value
    FROM annual_demand_raw
    GROUP BY 
        scenario,
        CASE WHEN segment IN ('transport_cars', 'transport_trucks', 'transport_rail') THEN 'transport' ELSE segment END,
        geography, 
        year
""")

# Show segments after aggregation
print("Segments after aggregation:")
print(con.execute("""
    SELECT segment, SUM(value) as total_gwh
    FROM annual_demand
    GROUP BY segment
    ORDER BY total_gwh DESC
""").fetchdf().to_string(index=False))

print(f"\nReduced from {con.execute('SELECT COUNT(*) FROM annual_demand_raw').fetchone()[0]:,} to "
      f"{con.execute('SELECT COUNT(*) FROM annual_demand').fetchone()[0]:,} rows")

Segments after aggregation:
    segment    total_gwh
   industry 8.285851e+06
    housing 3.022723e+06
   services 2.443331e+06
  transport 1.550036e+06
datacenters 5.418987e+05

Reduced from 11,466 to 8,190 rows


---

## Step 1: Create Profile Tables in DuckDB

Load all profiles into DuckDB tables for SQL-based profile extension.

### Profile Types:
- **Year-based** (housing, services, industry): Full 8760-hour CSV → lookup table by (month, weekday, hour)
- **Pattern-based** (transport, datacenters): JSON arrays → store as DuckDB arrays

In [9]:
# Load year-based profiles (housing, services, industry)
# Convert 8760-hour timeseries to (month, weekday, hour) lookup table
print("Loading year-based profiles...")

for segment in ['housing', 'services', 'industry']:
    cfg = SEGMENT_PROFILES[segment]
    file_path = PROFILE_PATH / cfg['file']
    
    # Load CSV with hour,value columns into DuckDB
    # The hour column is 0-8759, we need to map to (month, weekday, hour)
    # Using 2024 as reference year (leap year = 8784 hours, but these CSVs have 8760)
    con.execute(f"""
        CREATE OR REPLACE TABLE profile_{segment}_raw AS
        SELECT 
            row_number() OVER () - 1 as hour_index,
            value
        FROM read_csv('{file_path}', header=true)
    """)
    
    # Create lookup table: (month, weekday, hour) -> average profile value
    # This allows weekday-aware extension to future years
    con.execute(f"""
        CREATE OR REPLACE TABLE profile_{segment}_lookup AS
        WITH with_timestamp AS (
            SELECT
                hour_index,
                value,
                -- Map hour_index (0-8759) to timestamp in 2024
                TIMESTAMP '2024-01-01 00:00:00' + (hour_index * INTERVAL '1 hour') as ts
            FROM profile_{segment}_raw
        )
        SELECT
            EXTRACT(MONTH FROM ts)::INT as month,
            EXTRACT(ISODOW FROM ts)::INT as weekday,  -- 1=Monday, 7=Sunday
            EXTRACT(HOUR FROM ts)::INT as hour,
            AVG(value) as profile_value
        FROM with_timestamp
        GROUP BY month, weekday, hour
    """)
    
    count = con.execute(f"SELECT COUNT(*) FROM profile_{segment}_lookup").fetchone()[0]
    print(f"  ✓ {segment}: {count} lookup rows (month × weekday × hour)")

print("Year-based profiles loaded.")

Loading year-based profiles...
  ✓ housing: 2016 lookup rows (month × weekday × hour)
  ✓ services: 2016 lookup rows (month × weekday × hour)
  ✓ industry: 2016 lookup rows (month × weekday × hour)
Year-based profiles loaded.


In [10]:
# Load pattern-based profiles (transport_*, datacenters)
# These have hourly × weekday × monthly multiplier arrays
print("\nLoading pattern-based profiles...")

# Create table to store pattern profiles
con.execute("""
    CREATE TABLE profile_patterns (
        segment VARCHAR,
        hourly DOUBLE[],           -- 24 values (or NULL if separate weekday/weekend)
        hourly_weekday DOUBLE[],   -- 24 values for weekday (for rail, datacenters)
        hourly_weekend DOUBLE[],   -- 24 values for weekend (for rail)
        hourly_summer DOUBLE[],    -- 24 values for summer (for datacenters)
        weekday DOUBLE[],          -- 7 values (Mon-Sun), or NULL if dict format
        weekday_multiplier DOUBLE, -- For dict format: weekday multiplier
        weekend_multiplier DOUBLE, -- For dict format: weekend multiplier
        monthly DOUBLE[]           -- 12 values (Jan-Dec)
    )
""")

# Load each pattern profile
for segment in ['transport_cars', 'transport_trucks', 'transport_rail', 'datacenters']:
    cfg = SEGMENT_PROFILES[segment]
    file_path = PROFILE_PATH / cfg['file']
    
    with open(file_path) as f:
        pattern = json.load(f)
    
    # Extract hourly patterns
    hourly = pattern.get('hourly')
    hourly_weekday = pattern.get('hourly_weekday') or pattern.get('hourly_winter')
    hourly_weekend = pattern.get('hourly_weekend')
    hourly_summer = pattern.get('hourly_summer')
    
    # Extract weekday patterns (can be array or dict)
    weekday_raw = pattern.get('weekday', [1.0]*7)
    if isinstance(weekday_raw, dict):
        weekday_arr = None
        weekday_mult = weekday_raw.get('weekday', 1.0)
        weekend_mult = weekday_raw.get('weekend', 1.0)
    else:
        weekday_arr = weekday_raw
        weekday_mult = None
        weekend_mult = None
    
    # Extract monthly pattern
    monthly = pattern.get('monthly', [1.0]*12)
    
    # Insert into table
    con.execute("""
        INSERT INTO profile_patterns VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, [
        segment,
        hourly,
        hourly_weekday,
        hourly_weekend,
        hourly_summer,
        weekday_arr,
        weekday_mult,
        weekend_mult,
        monthly
    ])
    
    print(f"  ✓ {segment}: loaded pattern")

print("\nPattern profiles summary:")
print(con.execute("""
    SELECT 
        segment,
        CASE WHEN hourly IS NOT NULL THEN 'hourly' 
             WHEN hourly_weekday IS NOT NULL THEN 'hourly_weekday/weekend'
             ELSE 'unknown' END as hourly_type,
        CASE WHEN weekday IS NOT NULL THEN '7-array' 
             WHEN weekday_multiplier IS NOT NULL THEN 'dict'
             ELSE 'unknown' END as weekday_type
    FROM profile_patterns
""").fetchdf().to_string(index=False))


Loading pattern-based profiles...
  ✓ transport_cars: loaded pattern
  ✓ transport_trucks: loaded pattern
  ✓ transport_rail: loaded pattern
  ✓ datacenters: loaded pattern

Pattern profiles summary:
         segment            hourly_type weekday_type
  transport_cars                 hourly      7-array
transport_trucks                 hourly         dict
  transport_rail hourly_weekday/weekend      7-array
     datacenters hourly_weekday/weekend      7-array


---

## Step 2: Generate Timestamp Backbone

Create all hourly timestamps for 2025-2050 using DuckDB's `generate_series()`.

This generates ~228,000 timestamps instantly (compared to Python loop which would take seconds).

In [11]:
# Generate timestamp backbone for all years
print(f"Generating timestamps for {CONFIG['start_year']}-{CONFIG['end_year']}...")
start = time.time()

con.execute(f"""
    CREATE TABLE timestamps AS
    SELECT
        ts as timestamp,
        EXTRACT(YEAR FROM ts)::INT as year,
        EXTRACT(MONTH FROM ts)::INT as month,
        EXTRACT(ISODOW FROM ts)::INT as weekday,  -- 1=Monday, 7=Sunday (matches ISO standard)
        EXTRACT(HOUR FROM ts)::INT as hour,
        -- For datacenter summer detection
        CASE WHEN EXTRACT(MONTH FROM ts) IN (6, 7, 8) THEN true ELSE false END as is_summer,
        -- For weekday/weekend detection (weekday = 1-5, weekend = 6-7)
        CASE WHEN EXTRACT(ISODOW FROM ts) >= 6 THEN true ELSE false END as is_weekend
    FROM generate_series(
        TIMESTAMP '{CONFIG['start_year']}-01-01 00:00:00',
        TIMESTAMP '{CONFIG['end_year']}-12-31 23:00:00',
        INTERVAL '1 hour'
    ) t(ts)
""")

count = con.execute("SELECT COUNT(*) FROM timestamps").fetchone()[0]
years = con.execute("SELECT COUNT(DISTINCT year) FROM timestamps").fetchone()[0]
print(f"Generated {count:,} timestamps in {time.time()-start:.3f}s")
print(f"  Years: {years}")
print(f"  Hours per year: ~{count // years:,}")

Generating timestamps for 2025-2050...
Generated 227,904 timestamps in 0.046s
  Years: 26
  Hours per year: ~8,765


---

## Step 3: Build Extended Profiles via SQL

Extend profiles to cover all timestamps using SQL JOINs:

1. **Year-based profiles** (housing, services, industry): JOIN on (month, weekday, hour) for weekday-aware extension
2. **Pattern-based profiles**: Multiply hourly × weekday × monthly arrays
3. **Transport weighted profile**: Combine cars/trucks/rail by demand weight
4. **Normalize** each segment-year to sum to 1.0

In [12]:
# Extend year-based profiles (housing, services, industry)
# These use weekday-aware matching: (month, weekday, hour)
print("Extending year-based profiles...")
start = time.time()

con.execute("""
    CREATE TABLE extended_yearly_profiles AS
    -- Housing
    SELECT
        t.timestamp,
        t.year,
        'housing' as segment,
        p.profile_value as raw_value
    FROM timestamps t
    JOIN profile_housing_lookup p
        ON p.month = t.month
        AND p.weekday = t.weekday
        AND p.hour = t.hour
    UNION ALL
    -- Services
    SELECT
        t.timestamp,
        t.year,
        'services' as segment,
        p.profile_value as raw_value
    FROM timestamps t
    JOIN profile_services_lookup p
        ON p.month = t.month
        AND p.weekday = t.weekday
        AND p.hour = t.hour
    UNION ALL
    -- Industry
    SELECT
        t.timestamp,
        t.year,
        'industry' as segment,
        p.profile_value as raw_value
    FROM timestamps t
    JOIN profile_industry_lookup p
        ON p.month = t.month
        AND p.weekday = t.weekday
        AND p.hour = t.hour
""")

count = con.execute("SELECT COUNT(*) FROM extended_yearly_profiles").fetchone()[0]
print(f"Extended year-based profiles: {count:,} rows in {time.time()-start:.2f}s")

Extending year-based profiles...
Extended year-based profiles: 683,712 rows in 0.28s


In [13]:
# Extend transport sub-segment profiles (for weighted averaging)
# Each has different hourly/weekday format - handle all cases
print("Extending transport sub-profiles...")
start = time.time()

con.execute("""
    CREATE TABLE transport_sub_profiles AS
    SELECT
        t.timestamp,
        t.year,
        p.segment,
        -- Calculate raw value: hourly × weekday × monthly
        -- Handle different hourly patterns
        CASE
            -- transport_cars: has hourly array, weekday array
            WHEN p.segment = 'transport_cars' THEN
                p.hourly[t.hour + 1] * p.weekday[t.weekday] * p.monthly[t.month]
            -- transport_trucks: has hourly array, weekday dict (weekday_multiplier/weekend_multiplier)
            WHEN p.segment = 'transport_trucks' THEN
                p.hourly[t.hour + 1] 
                * (CASE WHEN t.is_weekend THEN p.weekend_multiplier ELSE p.weekday_multiplier END)
                * p.monthly[t.month]
            -- transport_rail: has hourly_weekday/hourly_weekend arrays, weekday array
            WHEN p.segment = 'transport_rail' THEN
                (CASE WHEN t.is_weekend THEN p.hourly_weekend[t.hour + 1] ELSE p.hourly_weekday[t.hour + 1] END)
                * p.weekday[t.weekday]
                * p.monthly[t.month]
        END as raw_value
    FROM timestamps t
    CROSS JOIN profile_patterns p
    WHERE p.segment IN ('transport_cars', 'transport_trucks', 'transport_rail')
""")

count = con.execute("SELECT COUNT(*) FROM transport_sub_profiles").fetchone()[0]
print(f"Extended transport sub-profiles: {count:,} rows in {time.time()-start:.2f}s")

Extending transport sub-profiles...
Extended transport sub-profiles: 683,712 rows in 0.11s


In [14]:
# Create weighted average transport profile
# Combines cars/trucks/rail by their relative demand weights
print("Creating weighted transport profile...")
start = time.time()

con.execute("""
    CREATE TABLE transport_profile_weighted AS
    SELECT
        tp.timestamp,
        tp.year,
        'transport' as segment,
        SUM(tp.raw_value * tw.weight) as raw_value
    FROM transport_sub_profiles tp
    JOIN transport_weights tw ON tp.segment = tw.segment
    GROUP BY tp.timestamp, tp.year
""")

count = con.execute("SELECT COUNT(*) FROM transport_profile_weighted").fetchone()[0]
print(f"Weighted transport profile: {count:,} rows in {time.time()-start:.2f}s")

Creating weighted transport profile...
Weighted transport profile: 227,904 rows in 0.08s


In [15]:
# Extend datacenter profile
# Datacenters have seasonal hourly patterns (summer vs winter)
print("Extending datacenter profile...")
start = time.time()

con.execute("""
    CREATE TABLE datacenter_profile AS
    SELECT
        t.timestamp,
        t.year,
        'datacenters' as segment,
        -- Use summer hourly pattern for Jun-Aug, winter for rest
        (CASE WHEN t.is_summer THEN p.hourly_summer[t.hour + 1] ELSE p.hourly_weekday[t.hour + 1] END)
        * p.weekday[t.weekday]
        * p.monthly[t.month] as raw_value
    FROM timestamps t
    CROSS JOIN profile_patterns p
    WHERE p.segment = 'datacenters'
""")

count = con.execute("SELECT COUNT(*) FROM datacenter_profile").fetchone()[0]
print(f"Datacenter profile: {count:,} rows in {time.time()-start:.2f}s")

Extending datacenter profile...
Datacenter profile: 227,904 rows in 0.02s


In [16]:
# Combine all profiles, add noise, and normalize per (segment, year) to sum to 1.0
# IMPORTANT: Noise is applied BEFORE normalization to guarantee exact annual totals
print("Combining profiles, adding noise, and normalizing...")
start = time.time()

# Set reproducible seed for noise
con.execute(f"SELECT setseed({CONFIG['noise_seed']})")

con.execute(f"""
    CREATE TABLE extended_profiles AS
    WITH combined AS (
        -- Year-based: housing, services, industry
        SELECT timestamp, year, segment, raw_value FROM extended_yearly_profiles
        UNION ALL
        -- Transport: weighted average of cars/trucks/rail
        SELECT timestamp, year, segment, raw_value FROM transport_profile_weighted
        UNION ALL
        -- Datacenters: seasonal pattern
        SELECT timestamp, year, segment, raw_value FROM datacenter_profile
    ),
    noisy AS (
        -- Apply ±2% noise to raw profile values (before normalization)
        SELECT
            timestamp,
            year,
            segment,
            raw_value * (1.0 + (random() - 0.5) * {CONFIG['noise_amplitude'] * 2}) as noisy_raw
        FROM combined
    )
    -- Normalize: noisy_raw / sum(noisy_raw for segment-year) = normalized value summing to 1.0
    -- This guarantees annual totals are exactly preserved
    SELECT
        timestamp,
        year,
        segment,
        noisy_raw / SUM(noisy_raw) OVER (PARTITION BY segment, year) as normalized_value
    FROM noisy
""")

count = con.execute("SELECT COUNT(*) FROM extended_profiles").fetchone()[0]
segments = con.execute("SELECT COUNT(DISTINCT segment) FROM extended_profiles").fetchone()[0]
print(f"Extended profiles: {count:,} rows ({segments} segments) in {time.time()-start:.2f}s")

# Verify normalization (should all be exactly 1.0)
print("\nVerifying normalization (should all be 1.0):")
print(con.execute("""
    SELECT segment, year, SUM(normalized_value) as sum_check
    FROM extended_profiles
    WHERE year IN (2025, 2030, 2040, 2050)
    GROUP BY segment, year
    ORDER BY segment, year
    LIMIT 20
""").fetchdf().to_string(index=False))

Combining profiles, adding noise, and normalizing...


Extended profiles: 1,139,520 rows (5 segments) in 0.33s

Verifying normalization (should all be 1.0):
    segment  year  sum_check
datacenters  2025        1.0
datacenters  2030        1.0
datacenters  2040        1.0
datacenters  2050        1.0
    housing  2025        1.0
    housing  2030        1.0
    housing  2040        1.0
    housing  2050        1.0
   industry  2025        1.0
   industry  2030        1.0
   industry  2040        1.0
   industry  2050        1.0
   services  2025        1.0
   services  2030        1.0
   services  2040        1.0
   services  2050        1.0
  transport  2025        1.0
  transport  2030        1.0
  transport  2040        1.0
  transport  2050        1.0


---

## Step 4: Join with Annual Demand

Join normalized profiles (with noise already applied) with annual demand values to create final hourly demand.

In [17]:
# Join profiles with annual demand to create final hourly demand
# Noise was already applied in cell-23 (before normalization), so no additional step needed
print("Joining profiles with annual demand...")
start = time.time()

con.execute("""
    CREATE TABLE hourly_demand AS
    SELECT
        a.scenario,
        a.segment,
        a.geography,
        e.timestamp,
        a.value * e.normalized_value as value  -- GWh × normalized profile = hourly GWh
    FROM annual_demand a
    JOIN extended_profiles e
        ON a.segment = e.segment
        AND a.year = e.year
""")

count = con.execute("SELECT COUNT(*) FROM hourly_demand").fetchone()[0]
print(f"Hourly demand: {count:,} rows in {time.time()-start:.2f}s")

Joining profiles with annual demand...
Hourly demand: 71,789,760 rows in 25.73s


In [18]:
# Validate: profiles sum to 1.0, so hourly sums equal annual totals by definition
mismatches = con.execute("""
    WITH hourly_sums AS (
        SELECT scenario, segment, geography, EXTRACT(YEAR FROM timestamp)::INT as year, SUM(value) as hourly_sum
        FROM hourly_demand
        GROUP BY scenario, segment, geography, EXTRACT(YEAR FROM timestamp)::INT
    )
    SELECT COUNT(*) FROM hourly_sums h
    JOIN annual_demand a ON h.scenario = a.scenario AND h.segment = a.segment AND h.geography = a.geography AND h.year = a.year
    WHERE ABS(h.hourly_sum - a.value) > 0.0001
""").fetchone()[0]
assert mismatches == 0, f"Validation failed: {mismatches} annual total mismatches"
print("✓ Validation passed: all annual totals preserved")

✓ Validation passed: all annual totals preserved


In [19]:
# Prepare output directory
import shutil

base_dir = OUTPUT_PATH / 'base'
if base_dir.exists():
    print(f"Removing existing base directory: {base_dir}")
    shutil.rmtree(base_dir)

print(f"Output path: {OUTPUT_PATH}")

Removing existing base directory: /home/viktor/code/behovskartan/generator/output/base
Output path: /home/viktor/code/behovskartan/generator/output


In [20]:
# Write segmented parquet files: base/{scenario}/{segment}/data.parquet
# This structure enables the API to load individual segments as needed
print("Writing segmented parquet files (one per scenario × segment)...")
start = time.time()

# Get unique scenarios and segments
scenarios = con.execute("SELECT DISTINCT scenario FROM hourly_demand ORDER BY scenario").fetchall()
segments = AGGREGATED_SEGMENTS  # ['housing', 'services', 'industry', 'transport', 'datacenters']

base_dir = OUTPUT_PATH / 'base'
total_rows = 0
files_created = 0

for (scenario_name,) in scenarios:
    scenario_path = base_dir / scenario_name
    scenario_path.mkdir(parents=True, exist_ok=True)
    
    for segment in segments:
        segment_path = scenario_path / segment
        segment_path.mkdir(parents=True, exist_ok=True)
        
        # Write parquet file for this scenario+segment
        con.execute(f"""
            COPY (
                SELECT 
                    timestamp,
                    value,
                    geography,
                    segment,
                    '{scenario_name}' as scenario_id
                FROM hourly_demand
                WHERE scenario = '{scenario_name}'
                  AND segment = '{segment}'
            )
            TO '{segment_path}/data.parquet'
            (FORMAT PARQUET, COMPRESSION '{CONFIG['compression']}')
        """)
        
        rows = con.execute(f"""
            SELECT COUNT(*) FROM hourly_demand 
            WHERE scenario = '{scenario_name}' AND segment = '{segment}'
        """).fetchone()[0]
        total_rows += rows
        files_created += 1
    
    print(f"  Written: base/{scenario_name}/ ({len(segments)} segments)")

elapsed = time.time() - start
print(f"\nTotal: {total_rows:,} rows in {files_created} files in {elapsed:.2f}s")
print(f"Structure: base/{'{scenario}'}/{'{segment}'}/data.parquet")

Writing segmented parquet files (one per scenario × segment)...


  Written: base/Beslutad Policy/ (5 segments)
  Written: base/Internationell Tillväxt/ (5 segments)
  Written: base/Lokal Miljöhänsyn/ (5 segments)

Total: 71,789,760 rows in 15 files in 30.57s
Structure: base/{scenario}/{segment}/data.parquet


In [21]:
# Generate pre-aggregated tables for fast queries
print("Generating aggregated tables...")
start = time.time()

aggregated_dir = OUTPUT_PATH / 'aggregated'
aggregated_dir.mkdir(exist_ok=True)

# 1. Yearly totals by geography (for map visualization)
print("  Creating geography_yearly.parquet...")
con.execute(f"""
    COPY (
        SELECT 
            scenario as scenario_id,
            geography,
            strftime(timestamp, '%Y') as year,
            SUM(value) as total_value
        FROM hourly_demand
        GROUP BY scenario, geography, strftime(timestamp, '%Y')
    ) TO '{aggregated_dir}/geography_yearly.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# 2. Yearly totals by segment (for sector charts)
print("  Creating segment_yearly.parquet...")
con.execute(f"""
    COPY (
        SELECT 
            scenario as scenario_id,
            segment,
            strftime(timestamp, '%Y') as year,
            SUM(value) as total_value
        FROM hourly_demand
        GROUP BY scenario, segment, strftime(timestamp, '%Y')
    ) TO '{aggregated_dir}/segment_yearly.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# 3. National yearly totals (for time series)
print("  Creating national_yearly.parquet...")
con.execute(f"""
    COPY (
        SELECT 
            scenario as scenario_id,
            strftime(timestamp, '%Y') as year,
            SUM(value) as total_value
        FROM hourly_demand
        GROUP BY scenario, strftime(timestamp, '%Y')
    ) TO '{aggregated_dir}/national_yearly.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

elapsed = time.time() - start
print(f"\nAggregated tables written in {elapsed:.2f}s")
print(f"  Location: {aggregated_dir}")

# Show output structure
print("\nOutput directory structure:")
import os
for root, dirs, files in os.walk(OUTPUT_PATH):
    if '__pycache__' in root:
        continue
    level = root.replace(str(OUTPUT_PATH), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for f in sorted(files)[:5]:
        print(f'{subindent}{f}')
    if len(files) > 5:
        print(f'{subindent}... and {len(files) - 5} more files')

Generating aggregated tables...
  Creating geography_yearly.parquet...
  Creating segment_yearly.parquet...
  Creating national_yearly.parquet...

Aggregated tables written in 10.02s
  Location: /home/viktor/code/behovskartan/generator/output/aggregated

Output directory structure:
output/
  _column_metadata.json
  aggregated/
    geography_yearly.parquet
    national_yearly.parquet
    segment_yearly.parquet
  base/
    Lokal Miljöhänsyn/
      services/
        data.parquet
      industry/
        data.parquet
      housing/
        data.parquet
      datacenters/
        data.parquet
      transport/
        data.parquet
    Beslutad Policy/
      services/
        data.parquet
      industry/
        data.parquet
      housing/
        data.parquet
      datacenters/
        data.parquet
      transport/
        data.parquet
    Internationell Tillväxt/
      services/
        data.parquet
      industry/
        data.parquet
      housing/
        data.parquet
      datacenters/
 

---

## Summary

In [22]:
# Record base generation end time
TIMING['base_end'] = time.time()

# Final summary
print("="*60)
print("BASE GENERATION COMPLETE")
print("="*60)

# Get stats
stats = con.execute("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT scenario) as scenarios,
        COUNT(DISTINCT segment) as segments,
        COUNT(DISTINCT geography) as geographies,
        MIN(timestamp) as min_ts,
        MAX(timestamp) as max_ts
    FROM hourly_demand
""").fetchone()

print(f"\nOutput Statistics:")
print(f"  Total rows: {stats[0]:,}")
print(f"  Scenarios: {stats[1]}")
print(f"  Segments: {stats[2]}")
print(f"  Geographies: {stats[3]}")
print(f"  Period: {stats[4]} to {stats[5]}")

# Show demand totals by scenario (all years)
print(f"\nTotal demand by scenario (all years, GWh):")
print(con.execute("""
    SELECT scenario, SUM(value) as total_gwh
    FROM hourly_demand
    GROUP BY scenario
    ORDER BY total_gwh DESC
""").fetchdf().to_string(index=False))

# Show demand by segment (first scenario)
print(f"\nDemand by segment (first scenario, all years, GWh):")
first_scenario = con.execute("SELECT scenario FROM hourly_demand LIMIT 1").fetchone()[0]
print(con.execute(f"""
    SELECT segment, SUM(value) as total_gwh
    FROM hourly_demand
    WHERE scenario = '{first_scenario}'
    GROUP BY segment
    ORDER BY total_gwh DESC
""").fetchdf().to_string(index=False))

# Show timing
if TIMING['base_start'] and TIMING['base_end']:
    base_elapsed = TIMING['base_end'] - TIMING['base_start']
    print(f"\n{'─'*60}")
    print(f"⏱  BASE GENERATION TIME: {base_elapsed:.1f}s ({base_elapsed/60:.1f} min)")
    print(f"{'─'*60}")

print(f"\nOutput path: {OUTPUT_PATH}")

BASE GENERATION COMPLETE

Output Statistics:
  Total rows: 71,789,760
  Scenarios: 3
  Segments: 5
  Geographies: 21
  Period: 2025-01-01 00:00:00 to 2050-12-31 23:00:00

Total demand by scenario (all years, GWh):
               scenario    total_gwh
Internationell Tillväxt 6.072632e+06
        Beslutad Policy 5.099209e+06
      Lokal Miljöhänsyn 4.671999e+06

Demand by segment (first scenario, all years, GWh):
    segment    total_gwh
   industry 2.260656e+06
    housing 1.013833e+06
   services 8.077193e+05
  transport 4.934917e+05
datacenters 9.630000e+04

────────────────────────────────────────────────────────────
⏱  BASE GENERATION TIME: 85.3s (1.4 min)
────────────────────────────────────────────────────────────

Output path: /home/viktor/code/behovskartan/generator/output


In [23]:
# Keep connection open for scenario generation
print("Base generation complete. Connection kept open for scenario variations.")

Base generation complete. Connection kept open for scenario variations.


---

## Scenario Variations

Generate percentage variations per segment with linear growth from 0% in 2025 to full percentage in 2050.

**Variations**: ±5%, ±10%, ±15% (6 variations per segment × 5 segments = 30 scenario variations)

In [24]:
# Scenario variation configuration
VARIATION_CONFIG = {
    'percentages': [-15, -10, -5, 5, 10, 15],  # Final year percentage change
    'segments': ['housing', 'services', 'industry', 'transport', 'datacenters'],
    'base_scenarios': ['Beslutad Policy'],  # Which base scenarios to use for variations
    'start_year': CONFIG['start_year'],  # 2025
    'end_year': CONFIG['end_year'],       # 2050
}

# Calculate total variations
n_variations = len(VARIATION_CONFIG['percentages']) * len(VARIATION_CONFIG['segments'])
n_base = len(VARIATION_CONFIG['base_scenarios'])
print(f"Scenario Variations Configuration:")
print(f"  Percentages: {VARIATION_CONFIG['percentages']}")
print(f"  Segments: {VARIATION_CONFIG['segments']}")
print(f"  Base scenarios: {VARIATION_CONFIG['base_scenarios']}")
print(f"  Total variations: {n_variations}")
print(f"  Rows per variation: {n_variations} × {n_base} base scenarios")
print(f"\nLinear growth formula:")
print(f"  multiplier(year) = 1.0 + (pct/100) × (year - {VARIATION_CONFIG['start_year']}) / ({VARIATION_CONFIG['end_year']} - {VARIATION_CONFIG['start_year']})")
print(f"\nExample for +15% housing:")
for year in [2025, 2030, 2040, 2050]:
    mult = 1.0 + (15/100) * (year - 2025) / 25
    print(f"  {year}: {mult:.3f} ({(mult-1)*100:+.1f}%)")

Scenario Variations Configuration:
  Percentages: [-15, -10, -5, 5, 10, 15]
  Segments: ['housing', 'services', 'industry', 'transport', 'datacenters']
  Base scenarios: ['Beslutad Policy']
  Total variations: 30
  Rows per variation: 30 × 1 base scenarios

Linear growth formula:
  multiplier(year) = 1.0 + (pct/100) × (year - 2025) / (2050 - 2025)

Example for +15% housing:
  2025: 1.000 (+0.0%)
  2030: 1.030 (+3.0%)
  2040: 1.090 (+9.0%)
  2050: 1.150 (+15.0%)


In [25]:
# Generate scenario variations
# For each segment and percentage, apply linear growth multiplier
print("Generating scenario variations...")
TIMING['scenarios_start'] = time.time()
TIMING['scenario_times'] = []  # Reset scenario times

scenarios_dir = OUTPUT_PATH / 'scenarios'
if scenarios_dir.exists():
    print(f"Removing existing scenarios directory: {scenarios_dir}")
    shutil.rmtree(scenarios_dir)
scenarios_dir.mkdir(parents=True, exist_ok=True)

start_year = VARIATION_CONFIG['start_year']
end_year = VARIATION_CONFIG['end_year']
year_span = end_year - start_year  # 25 years

# Build WHERE clause for base scenario filter
base_scenarios = VARIATION_CONFIG['base_scenarios']
base_filter = "scenario IN (" + ", ".join(f"'{s}'" for s in base_scenarios) + ")"
print(f"Using base scenarios: {base_scenarios}")

total_variations = 0

for segment in VARIATION_CONFIG['segments']:
    for pct in VARIATION_CONFIG['percentages']:
        start = time.time()
        
        # Create scenario name: segment_+15 or segment_-10
        scenario_name = f"{segment}_{pct:+d}"
        scenario_path = scenarios_dir / scenario_name
        scenario_path.mkdir(parents=True, exist_ok=True)
        
        # Apply linear multiplier to just this segment
        # multiplier = 1.0 + (pct/100) * (year - start_year) / year_span
        # Only include selected base scenarios
        con.execute(f"""
            COPY (
                SELECT 
                    timestamp,
                    CASE 
                        WHEN segment = '{segment}' 
                        THEN value * (1.0 + ({pct}/100.0) * (EXTRACT(YEAR FROM timestamp) - {start_year}) / {year_span})
                        ELSE value
                    END as value,
                    geography,
                    segment,
                    scenario as scenario_id
                FROM hourly_demand
                WHERE {base_filter}
            )
            TO '{scenario_path}/data.parquet'
            (FORMAT PARQUET, COMPRESSION '{CONFIG['compression']}')
        """)
        
        elapsed = time.time() - start
        total_variations += 1
        TIMING['scenario_times'].append((scenario_name, elapsed))
        print(f"  ✓ {scenario_name} ({elapsed:.1f}s)")

TIMING['scenarios_end'] = time.time()
scenarios_elapsed = TIMING['scenarios_end'] - TIMING['scenarios_start']

# Calculate timing statistics
times = [t for _, t in TIMING['scenario_times']]
min_time = min(times)
max_time = max(times)
avg_time = sum(times) / len(times)
min_scenario = next(name for name, t in TIMING['scenario_times'] if t == min_time)
max_scenario = next(name for name, t in TIMING['scenario_times'] if t == max_time)

print(f"\n{'='*60}")
print(f"SCENARIO VARIATIONS COMPLETE")
print(f"{'='*60}")
print(f"  Total variations: {total_variations}")
print(f"  Base scenarios used: {base_scenarios}")
print(f"  Output: {scenarios_dir}")
print(f"\n{'─'*60}")
print(f"⏱  SCENARIO TIMING STATISTICS")
print(f"{'─'*60}")
print(f"  Total time:   {scenarios_elapsed:.1f}s ({scenarios_elapsed/60:.1f} min)")
print(f"  Min time:     {min_time:.1f}s ({min_scenario})")
print(f"  Max time:     {max_time:.1f}s ({max_scenario})")
print(f"  Avg time:     {avg_time:.1f}s per scenario")
print(f"{'─'*60}")

# Show combined timing if base was also generated
if TIMING['base_start'] and TIMING['base_end']:
    base_elapsed = TIMING['base_end'] - TIMING['base_start']
    total_elapsed = base_elapsed + scenarios_elapsed
    print(f"\n{'='*60}")
    print(f"⏱  TOTAL GENERATION TIME SUMMARY")
    print(f"{'='*60}")
    print(f"  Base scenarios:      {base_elapsed:>8.1f}s ({base_elapsed/60:.1f} min)")
    print(f"  Scenario variations: {scenarios_elapsed:>8.1f}s ({scenarios_elapsed/60:.1f} min)")
    print(f"  {'─'*40}")
    print(f"  TOTAL:               {total_elapsed:>8.1f}s ({total_elapsed/60:.1f} min)")
    print(f"{'='*60}")

Generating scenario variations...
Removing existing scenarios directory: /home/viktor/code/behovskartan/generator/output/scenarios


Using base scenarios: ['Beslutad Policy']
  ✓ housing_-15 (6.5s)
  ✓ housing_-10 (3.3s)
  ✓ housing_-5 (5.2s)
  ✓ housing_+5 (3.3s)
  ✓ housing_+10 (2.8s)
  ✓ housing_+15 (2.8s)
  ✓ services_-15 (18.2s)
  ✓ services_-10 (4.9s)
  ✓ services_-5 (4.0s)
  ✓ services_+5 (3.3s)
  ✓ services_+10 (3.1s)
  ✓ services_+15 (3.1s)
  ✓ industry_-15 (3.1s)
  ✓ industry_-10 (3.5s)
  ✓ industry_-5 (2.9s)
  ✓ industry_+5 (3.3s)
  ✓ industry_+10 (3.7s)
  ✓ industry_+15 (4.7s)
  ✓ transport_-15 (4.2s)
  ✓ transport_-10 (3.5s)
  ✓ transport_-5 (3.2s)
  ✓ transport_+5 (2.8s)
  ✓ transport_+10 (3.6s)
  ✓ transport_+15 (3.3s)
  ✓ datacenters_-15 (3.2s)
  ✓ datacenters_-10 (3.0s)
  ✓ datacenters_-5 (3.3s)
  ✓ datacenters_+5 (3.1s)
  ✓ datacenters_+10 (2.7s)
  ✓ datacenters_+15 (2.8s)

SCENARIO VARIATIONS COMPLETE
  Total variations: 30
  Base scenarios used: ['Beslutad Policy']
  Output: /home/viktor/code/behovskartan/generator/output/scenarios

────────────────────────────────────────────────────────────
⏱  

---

## Independent Parameter Generation (Strategy 2)

Generate segment-only parquet files for independent parameters.

**Structure**: `parameters/{parameter}/{index}/{segment}/data.parquet`

This allows the API to combine parameter files at query time:
- For `housing_growth=2, transport_growth=0`:
  - Read `parameters/housing_growth/2/housing/data.parquet`
  - Read `base/{scenario}/transport/data.parquet` (index 0 = baseline)

Parameters are defined in `config.yaml` under the `parameters` section.

In [26]:
# Load parameter definitions from config.yaml
import yaml

# Load config
CONFIG_PATH = GENERATOR_PATH.parent / 'config.yaml'
with open(CONFIG_PATH) as f:
    project_config = yaml.safe_load(f)

param_config = project_config.get('parameters', {})
param_definitions = param_config.get('definitions', {})

print(f"Loaded config from: {CONFIG_PATH}")
print(f"\nParameter Configuration:")
print(f"  Strategy: {param_config.get('strategy', 'N/A')}")
print(f"  Base scenario: {param_config.get('baseScenario', 'N/A')}")
print(f"  Parameters defined: {len(param_definitions)}")

for name, defn in param_definitions.items():
    segments = defn.get('segments', [])
    values = [v['index'] for v in defn.get('values', []) if v['index'] > 0]
    print(f"    {name}: segments={segments}, indices={values}")

Loaded config from: /home/viktor/code/behovskartan/config.yaml

Parameter Configuration:
  Strategy: 2
  Base scenario: Beslutad Policy
  Parameters defined: 10
    housing_growth: segments=['housing'], indices=[1, 2, 3]
    housing_flex: segments=['housing'], indices=[1, 2]
    transport_growth: segments=['transport'], indices=[1, 2, 3]
    transport_flex: segments=['transport'], indices=[1, 2]
    industry_growth: segments=['industry'], indices=[1, 2, 3]
    industry_flex: segments=['industry'], indices=[1, 2]
    services_growth: segments=['services'], indices=[1, 2, 3]
    services_flex: segments=['services'], indices=[1, 2]
    datacenters_growth: segments=['datacenters'], indices=[1, 2, 3]
    datacenters_flex: segments=['datacenters'], indices=[1, 2]


In [27]:
# Validate all curve files exist before parameter generation
print("Validating curve files...")

# Define project_root for path resolution
project_root = GENERATOR_PATH.parent

missing_files = []
required_files = []

for param_name, param_def in param_definitions.items():
    for value_def in param_def.get('values', []):
        curve_def = value_def.get('curve')
        if curve_def is None:
            continue  # Index 0, no file needed

        curve_path = project_root / curve_def['file']
        required_files.append((param_name, value_def['index'], curve_path))

        if not curve_path.exists():
            missing_files.append((param_name, value_def['index'], curve_path))

print(f"Required curve files: {len(required_files)}")
print(f"Missing curve files: {len(missing_files)}")

if missing_files:
    print("\n" + "="*60)
    print("ERROR: Missing curve files!")
    print("="*60)
    print("\nThe following curve files are required but not found:\n")
    for param, idx, path in missing_files:
        print(f"  - {param} index {idx}: {path.name}")
        print(f"    Path: {path}")

    print("\n" + "-"*60)
    print("TO FIX: Run the generate_curves.ipynb notebook in each folder:")
    print("-"*60)
    folders = set(p.parent for _, _, p in missing_files)
    for folder in sorted(folders):
        print(f"  - {folder}/generate_curves.ipynb")

    print("\n" + "="*60)
    raise FileNotFoundError(f"Missing {len(missing_files)} curve file(s). See above for details.")
else:
    print("✓ All curve files present")

Validating curve files...
Required curve files: 25
Missing curve files: 0
✓ All curve files present


In [28]:
# Generate parameter files for Strategy 2 (Independent Parameters)
# Output: parameters/{param}/{index}/{segment}/data.parquet
print("Generating independent parameter files...")
TIMING['params_start'] = time.time()
TIMING['param_times'] = []

base_scenario = param_config.get('baseScenario', 'Beslutad Policy')
params_dir = OUTPUT_PATH / 'parameters'

# Remove existing parameters directory
if params_dir.exists():
    print(f"Removing existing parameters directory: {params_dir}")
    shutil.rmtree(params_dir)
params_dir.mkdir(parents=True, exist_ok=True)

total_files = 0
project_root = GENERATOR_PATH.parent

for param_name, param_def in param_definitions.items():
    param_start = time.time()
    
    segments = param_def.get('segments', [])
    values = param_def.get('values', [])
    how = param_def.get('how', 'multiply')
    
    files_for_param = 0
    
    for value_def in values:
        idx = value_def.get('index')
        curve_def = value_def.get('curve')
        
        # Skip index 0 (baseline - use base data directly)
        if idx == 0 or curve_def is None:
            continue
        
        # Load curve file
        curve_file = project_root / curve_def['file']
        if not curve_file.exists():
            print(f"  ⚠ Curve file not found: {curve_file}")
            continue
        
        filter_expr = curve_def.get('filter', '1=1')
        
        for segment in segments:
            # Path to base segment data
            base_segment_path = OUTPUT_PATH / 'base' / base_scenario / segment / 'data.parquet'
            
            if not base_segment_path.exists():
                print(f"  ⚠ Base segment not found: {base_segment_path}")
                continue
            
            # Output path: parameters/{param}/{index}/{segment}/data.parquet
            out_path = params_dir / param_name / str(idx) / segment
            out_path.mkdir(parents=True, exist_ok=True)
            out_file = out_path / 'data.parquet'
            
            # Build SQL to apply curve
            if how == 'multiply':
                value_expr = "b.value * COALESCE(c.value, 1.0)"
            else:  # add
                value_expr = "b.value + COALESCE(c.value, 0.0)"
            
            sql = f"""
            COPY (
                WITH curve_data AS (
                    SELECT timestamp, value
                    FROM read_parquet('{curve_file}')
                    WHERE {filter_expr}
                )
                SELECT
                    b.timestamp,
                    {value_expr} AS value,
                    b.geography,
                    b.segment
                FROM read_parquet('{base_segment_path}') b
                LEFT JOIN curve_data c ON b.timestamp = c.timestamp
            )
            TO '{out_file}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
            """
            
            con.execute(sql)
            files_for_param += 1
            total_files += 1
    
    elapsed = time.time() - param_start
    TIMING['param_times'].append((param_name, elapsed, files_for_param))
    print(f"  ✓ {param_name}: {files_for_param} files ({elapsed:.1f}s)")

TIMING['params_end'] = time.time()
params_elapsed = TIMING['params_end'] - TIMING['params_start']

print(f"\n{'='*60}")
print(f"PARAMETER GENERATION COMPLETE")
print(f"{'='*60}")
print(f"  Total parameters: {len(param_definitions)}")
print(f"  Total files: {total_files}")
print(f"  Output: {params_dir}")
print(f"\n{'─'*60}")
print(f"⏱  PARAMETER TIMING: {params_elapsed:.1f}s ({params_elapsed/60:.1f} min)")
print(f"{'─'*60}")

Generating independent parameter files...
Removing existing parameters directory: /home/viktor/code/behovskartan/generator/output/parameters
  ✓ housing_growth: 3 files (1.2s)
  ✓ housing_flex: 2 files (0.7s)
  ✓ transport_growth: 3 files (1.1s)
  ✓ transport_flex: 2 files (0.7s)
  ✓ industry_growth: 3 files (1.1s)
  ✓ industry_flex: 2 files (0.8s)
  ✓ services_growth: 3 files (1.3s)
  ✓ services_flex: 2 files (0.8s)
  ✓ datacenters_growth: 3 files (1.2s)
  ✓ datacenters_flex: 2 files (0.8s)

PARAMETER GENERATION COMPLETE
  Total parameters: 10
  Total files: 25
  Output: /home/viktor/code/behovskartan/generator/output/parameters

────────────────────────────────────────────────────────────
⏱  PARAMETER TIMING: 9.7s (0.2 min)
────────────────────────────────────────────────────────────
